<a href="https://colab.research.google.com/github/kavinraajs04/DeepLearning/blob/main/Experiment%204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install and load the AG News dataset from Hugging Face.

# Import the required libraries for data loading and manipulation.

In [1]:
!pip install dataset

from datasets import load_dataset
import pandas as pd

raw_dataset = load_dataset("wangrongsheng/ag_news")
df=pd.DataFrame(raw_dataset['train'])
print(df.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 12.1 MB/s eta 0:00:00


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


# Load the AG News dataset and convert the training data into a Pandas DataFrame.

# Step 2: Clean and normalize the text by converting it to lowercase and removing special characters.

In [2]:
import re

def clean_text(text):
    text = text.lower()                          # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)      # Remove numbers and special characters
    return text

df['clean_text'] = df['text'].apply(clean_text)

print(df[['text', 'clean_text']].head())

                                                text  \
0  Wall St. Bears Claw Back Into the Black (Reute...   
1  Carlyle Looks Toward Commercial Aerospace (Reu...   
2  Oil and Economy Cloud Stocks' Outlook (Reuters...   
3  Iraq Halts Oil Exports from Main Southern Pipe...   
4  Oil prices soar to all-time record, posing new...   

                                          clean_text  
0  wall st bears claw back into the black reuters...  
1  carlyle looks toward commercial aerospace reut...  
2  oil and economy cloud stocks outlook reuters r...  
3  iraq halts oil exports from main southern pipe...  
4  oil prices soar to alltime record posing new m...  


# Step 3: Convert words into integer sequences using the Keras Tokenizer.

In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['clean_text'])

sequences = tokenizer.texts_to_sequences(df['clean_text'])

print(sequences[:2])   # Display first 2 tokenized articles

[[391, 324, 1525, 14260, 99, 54, 1, 812, 23, 23, 38863, 391, 1988, 50537, 4, 38864, 34, 3893, 737, 295], [15257, 1004, 801, 1243, 4132, 23, 23, 875, 745, 302, 15257, 50538, 21, 3, 4315, 8, 523, 38865, 6, 50539, 2041, 5, 1, 480, 226, 21, 3765, 50540, 6380, 7, 182, 327, 4, 1, 112]]


# Step 4: Pad or truncate all text sequences to a fixed length of 50 words.

In [4]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 50

X = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

print(X.shape)

(120000, 50)


# Step 5: Convert class labels into one-hot encoded vectors for multi-class classification.

In [5]:
from tensorflow.keras.utils import to_categorical

y = to_categorical(df['label'])

print(y[:5])

[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


# Step 6: Build the RNN model using Embedding, SimpleRNN, and Dense layers.

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

vocab_size = len(tokenizer.word_index) + 1

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=50),
    SimpleRNN(64),
    Dense(4, activation='softmax')   # AG News has 4 classes
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Step 7: Compile the model using the Adam optimizer and categorical cross-entropy loss.

In [7]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
history = model.fit(
    X,
    y,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 25s 7ms/step - accuracy: 0.8102 - loss: 0.5430 - val_accuracy: 0.8480 - val_loss: 0.4410
Epoch 2/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.9022 - loss: 0.3100 - val_accuracy: 0.8608 - val_loss: 0.4432
Epoch 3/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.8693 - loss: 0.4038 - val_accuracy: 0.7185 - val_loss: 0.8116
Epoch 4/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.6677 - loss: 0.9067 - val_accuracy: 0.3776 - val_loss: 1.2292
Epoch 5/5
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.4412 - loss: 1.1548 - val_accuracy: 0.4150 - val_loss: 1.2090


In [9]:
loss, accuracy = model.evaluate(X, y)

print("Loss:", loss)
print("Accuracy:", accuracy)

3750/3750 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.4396 - loss: 1.1544
Loss: 1.1543631553649902
Accuracy: 0.4395916759967804


In [10]:
sample_text = ["Apple launches a new AI-powered iPhone with advanced features."]

sample_text = [clean_text(text) for text in sample_text]

sample_seq = tokenizer.texts_to_sequences(sample_text)

sample_pad = pad_sequences(sample_seq, maxlen=50, padding='post', truncating='post')

prediction = model.predict(sample_pad)

predicted_class = prediction.argmax(axis=1)[0]

print("Predicted Class:", predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
Predicted Class: 3
